# Entropy Analysis for Signal Complexity

**Dataset**: PhysioNet Auditory EEG  
**Channels**: P4, Cz, F8, T7  
**Sampling rate**: 200 Hz  
**Subject**: 1

---

## Overview

Entropy measures the complexity or uncertainty in a signal. We compute three types: sample entropy, approximate entropy, and spectral entropy on sliding windows of channel P4.

## Expected outputs

- Three plots: original signal, sample/approximate entropy, spectral entropy
- Fluctuation in entropy values reflecting changing signal complexity over time

## Key parameters

| Parameter | Value | Meaning |
| --- | --- | --- |
| Channel | P4 | Parietal region |
| WINDOW | 1000 | 5 second window |
| STEP | 500 | 2.5 second step |
| order | 2 | Embedding dimension |
| tolerance | 0.2*std | Similarity threshold |


## 1. Install dependencies


In [ ]:
!pip install scipy numpy plotly antropy mne wfdb


## 2. Clone repo and download data

We download only subject 1 (`--subjects 1`) to speed up the experiment in Colab.


In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


## 3. Load the EEG signal

We load subject 1, experiment 1, session 2.


In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
fs = 200

print(f'Channels: {ch_names}')
print(f'Signal length: {len(eeg_data)} samples ({len(eeg_data)/fs:.1f} seconds)')


## 4. Apply entropy analysis

We compute three entropy measures on sliding windows of channel P4.


In [ ]:
from antropy import sample_entropy, app_entropy, spectral_entropy

WINDOW = 1000
STEP = 500
channel_data = eeg_data[:, 0]

n_windows = (len(channel_data) - WINDOW) // STEP + 1
sampen_vals = []
appen_vals = []
specen_vals = []
window_centers = []

for i in range(n_windows):
    start = i * STEP
    end = start + WINDOW
    segment = np.ascontiguousarray(channel_data[start:end])
    r_val = 0.2 * np.std(segment)
    sampen_vals.append(sample_entropy(segment, order=2, tolerance=r_val))
    appen_vals.append(app_entropy(segment, order=2, tolerance=r_val))
    specen_vals.append(spectral_entropy(segment, sf=fs, method='welch', normalize=True))
    window_centers.append((start + end) / 2 / fs)

print(f'Computed {n_windows} windows')


## 5. Interactive plot

**What to look for:**

- Regions with higher complexity show higher entropy
- Sample and approximate entropy generally track each other
- Spectral entropy measures energy distribution across frequencies



In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

n_plot = min(5000, len(channel_data))
t_sig = np.arange(n_plot) / fs

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=('Original signal - Channel P4',
                                    'Sample and Approximate Entropy',
                                    'Spectral Entropy (normalized)'))
fig.add_trace(go.Scatter(x=t_sig, y=channel_data[:n_plot], name='Signal',
                         line=dict(color='blue', width=0.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=window_centers, y=sampen_vals, name='Sample Entropy',
                         line=dict(color='green', width=1.5)), row=2, col=1)
fig.add_trace(go.Scatter(x=window_centers, y=appen_vals, name='Approximate Entropy',
                         line=dict(color='orange', width=1.5)), row=2, col=1)
fig.add_trace(go.Scatter(x=window_centers, y=specen_vals, name='Spectral Entropy',
                         line=dict(color='purple', width=1.5)), row=3, col=1)
fig.update_layout(height=900, title_text='Entropy Analysis - Channel P4',
                  xaxis3_title='Time (s)', showlegend=True)
fig.show()


## What did we learn?

- Entropy measures signal complexity without machine learning
- Sample entropy is a popular tool for monitoring mental stress
- Spectral entropy reveals energy distribution across frequencies
- Sliding windows reveal temporal changes in complexity

